# 6.4 量化方式：粒度、组合与随路量化

## 1. 量化粒度（量化模式）

设左矩阵（激活 activation，shape (m, k)）× 右矩阵（权重 weight，shape (k, n)），k 为 reduce 轴。**量化模式**指对输入 Tensor 采用的量化计算级别：

| 模式 | 简称 | 量化对象 | scale shape | 说明 |
|--|--|--|--|--|
| pertensor | T | 左或右 | (1, ) | 整个 Tensor 一个 scale |
| perchannel | C | 右矩阵 | (n, ) | 每个 channel 一个 |
| pertoken | K | 左矩阵 | (m, ) | 每个 token 一个 |
| pergroup | G | 左或右 | (m, k/gs) / (k/gs, n) | **reduce 轴上分组**，每组一个 |
| perblock | B | 左或右 | (m/bs, k/bs) / (k/bs, n/bs) | 所有轴上分块 |

粒度越细 → 精度越好，但 scale 的存储、搬运与施加开销越大。**粒度选择就是「精度 ↔ scale 开销」的权衡**。

## 2. 全量化组合模式

对左、右矩阵均量化的模式叫**全量化**：

| 组合 | 名称 | 典型场景 |
|--|--|--|
| T-C | pertensor-perchannel | 激活按 tensor、权重按 channel |
| K-C | pertoken-perchannel | 激活按 token、权重按 channel |
| G-B | pergroup-perblock | 激活按组、权重按块 |
| T-CG | pertensor-perchannel-pergroup | 例：A8W4 伪量化——A 是 pertensor，weight 转 W8 是 perchannel，W8 转浮点是 pergroup |
| B-B | perblock-perblock | 双侧分块 |

## 3. mx 量化：G-G 的特例

**mx 量化的本质是 pergroup-pergroup（G-G）量化模式**，且有两个特殊限定：

1. scale 的数据类型为 **FLOAT8_E8M0**（只能表示 2 的幂，见 6.2 节）；
2. **group size 固定为 32**。

即：数据每 32 个元素一组，每组一个 E8M0 scale。**MXFP8 = E4M3 数据 + E8M0 scale（gs=32）**，这就是它在 6.5 节 FA 方案里的存储形态。

## 4. 随路量化（on-the-fly）

独立部署 Quant/AntiQuant 算子意味着「GM 读出 → 量化/反量化 → 写回 GM → 下一个算子再读出」的额外往返。**随路量化把这一步搬进消费算子的 kernel 内**：数据在搬入/计算前就地完成量化或反量化，省掉一次 GM 读写往返。

- 对 KV Cache：写入时量化一次（离线或 ingress 算子），FA kernel 内**随路反量化**——这是 6.5 节 MXFP8 方案的形态；
- scale 为 E8M0 时，随路反量化退化为「指数加 / 移位」，代价极低，这正是 mx 设计的巧妙之处。

## 小测验

1. 五种量化粒度 T/C/K/G/B 的 scale shape 分别是什么？哪个粒度的 scale 开销最大？
2. mx 量化在 G-G 模式上加了哪两条特殊限定？这两条分别换来了什么？
3. 什么是随路量化？相比独立 Quant/AntiQuant 算子，它省掉了什么、又要求算子自己做什么？
4. pergroup 量化时，scale 沿 reduce 轴分组。这给矩阵乘的 scale 施加带来什么麻烦（提示：分组累加）？

In [ ]:
# 动手：粒度对精度的影响——为什么需要 pergroup（mx）
import numpy as np

np.random.seed(42)
m, k = 8, 128
act = np.random.randn(m, k).astype(np.float32) * 0.1
act[:, 0] = 12.0        # 注入离群通道：一个通道的 amax 是其他通道的 ~100 倍
QMAX = 448.0            # E4M3 的最大值

def quant_dequant(x, scale_per_group=None, gs=None):
    # 对称量化-反量化。scale_per_group=True 时沿 k 轴按 gs 分组。
    x = x.copy()
    if scale_per_group:
        for g0 in range(0, x.shape[-1], gs):
            blk = x[..., g0:g0+gs]
            s = np.abs(blk).max() / QMAX
            x[..., g0:g0+gs] = np.round(blk / s) * s
    else:
        s = np.abs(x).max() / QMAX
        x = np.round(x / s) * s
    return x

pt = quant_dequant(act)                       # pertensor
pg = quant_dequant(act, True, gs=32)          # pergroup，gs=32（mx 风格）

def rel_err(a, b):
    return np.abs(a - b).mean() / np.abs(a).mean()

print(f'pertensor  相对误差: {rel_err(act, pt):.3%}')
print(f'pergroup32 相对误差: {rel_err(act, pg):.3%}')
print()
print('观察：离群通道把全局 scale 拉大 ~100 倍，其他通道在 pertensor 下几乎全部')
print('     舍入到 0；pergroup 把离群关进自己那组，其他组仍用小 scale 保持分辨率。')
print('     这就是 MXFP8 用 gs=32 块 scale 的根本原因——离群值只污染自己的组。')